<a href="https://colab.research.google.com/github/DomiDelarosa/analisis-datos-python/blob/main/ExamenBD2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Cargar biblioteca Pandas**
---


In [ ]:
import pandas as pd

## Cargar datos


Uso la biblioteca gdown para descargar archivos hacia el punto local de Colab

Link para descargar base de datos: [MiTiendaOnline.xlsx](https://docs.google.com/spreadsheets/d/1AjnUBZHEN5psSbTI73ah-ya5WkFDDK8q/edit?usp=drive_link&ouid=114276166454003590074&rtpof=true&sd=true)

In [ ]:
import gdown

file_id = '16OK-L-j66wU3RJtygqjEKqVAQqPzGyAB'
url = f'https://drive.google.com/uc?id={file_id}'

gdown.download(url, 'MiTiendaOnline.xlsx', quiet=False)

df = pd.ExcelFile('MiTiendaOnline.xlsx') # abrir archivo para revisar hojas (confirmar que funciona el archivo)
print(df.sheet_names)

Downloading...
From: https://drive.google.com/uc?id=16OK-L-j66wU3RJtygqjEKqVAQqPzGyAB
To: /content/MiTiendaOnline.xlsx
100%|██████████| 94.7k/94.7k [00:00<00:00, 35.7MB/s]


['Hoja1', 'Clientes', 'Empleados', 'Facturas', 'Pedidos', 'Productos', 'Proveedores']


---
## Leer tablas de la base de datos
---

In [ ]:
try:
  clientes = pd.read_excel('MiTiendaOnline.xlsx', sheet_name='Clientes')
  facturas = pd.read_excel('MiTiendaOnline.xlsx', sheet_name='Facturas')
  productos = pd.read_excel('MiTiendaOnline.xlsx', sheet_name='Productos')

  print("\nDataFrame cargado desde hojas específicas de Excel: Clientes, Facturas y Productos.")

except FileNotFoundError:
  print("El archivo 'MiTiendaOnline.xlsx' no se encontró.")

except Exception as e:
  print(f"Ocurrió un error al cargar la hoja de Excel: {e}")


DataFrame cargado desde hojas específicas de Excel: Clientes, Facturas y Productos.


---
## Hacer consultas


### **1.** Mostrar la información de los clientes que hayan hecho las mayores compras (compras por encima del promedio de las cantidades)

In [ ]:
total_productos_comprados = facturas.groupby('id_cliente')['cantidad_productos'].sum()

promedio_cantidades = total_productos_comprados.mean()

print('Promedio de cantidades compradas: ', round(promedio_cantidades, 2), "\n")

clientes_mayor_compra = total_productos_comprados[total_productos_comprados > promedio_cantidades]

info_clientes_cantidades = pd.merge(clientes_mayor_compra, clientes, on='id_cliente')

infoTotal = info_clientes_cantidades[['id_cliente', 'nombre_cliente', 'apellido_cliente', 'cantidad_productos']].sort_values('cantidad_productos', ascending=True)
pd.DataFrame(infoTotal)

Promedio de cantidades compradas:  665.07 



,id_cliente,nombre_cliente,apellido_cliente,cantidad_productos
2,3,Javier,Rodríguez,699
8,20,Natalia,Morales,715
5,9,Felipe,Hernández,759
6,12,Lucía,Moreno,829
4,6,Isabella,Martínez,855
0,1,Alejandro,García,866
3,5,Daniel,Pérez,925
1,2,Valeria,López,952
9,22,Andrea,Núñez,965
11,30,Laura,Cortés,967


---
### **2.** Mostrar la información de los clientes  que hayan hecho las mayores compras (en cantidad de dinero, osea, cantidad de productos x precio unitario por encima del promedio).

In [ ]:
facturas['total_compra'] = (facturas['cantidad_productos'] * facturas['precio_unitario'])
total_comprado = facturas.groupby('id_cliente')['total_compra'].sum()

promedio_compras = total_comprado.mean()

print('Promedio de cantidades compradas: ', round(promedio_compras, 2), "\n")

clientes_mayor_comprado = total_comprado[total_comprado > promedio_compras]

info_clientes = pd.merge(clientes_mayor_comprado, clientes, on='id_cliente')

infoTotal = info_clientes[['id_cliente', 'nombre_cliente', 'apellido_cliente', 'total_compra']].sort_values('total_compra', ascending=True)
pd.DataFrame(infoTotal)

Promedio de cantidades compradas:  319816.13 



,id_cliente,nombre_cliente,apellido_cliente,total_compra
0,1,Alejandro,García,346843.0
3,5,Daniel,Pérez,358274.0
9,13,Andrés,Álvarez,358423.0
2,3,Javier,Rodríguez,367004.0
11,20,Natalia,Morales,368799.0
7,11,Pablo,Díaz,380899.0
8,12,Lucía,Moreno,389586.0
5,9,Felipe,Hernández,391824.0
13,25,Leonardo,Iglesias,393608.0
6,10,Gabriela,Jiménez,399417.0


---
### **3.** Mostrar toda la información del mejor cliente. El primero en el caso que haya más de uno.

In [ ]:
facturas['total_compra'] = (facturas['cantidad_productos'] * facturas['precio_unitario'])

total_comprado = facturas.groupby('id_cliente')['total_compra'].sum()

mayor_compra = total_comprado.max()

mejor_cliente = total_comprado[total_comprado == mayor_compra]

info_mejor_cliente = pd.merge(mejor_cliente, clientes, on='id_cliente')

pd.DataFrame(info_mejor_cliente.head(1))

,id_cliente,total_compra,nombre_cliente,apellido_cliente,direccion_cliente,telefono_cliente,fecha_creacion
0,18,621441.0,Camila,Vargas,"Carrer Major 3, Palma de Mallorca",34678901234,2018-03-02


---
### **4.** Muestre la información del producto que posee el mejor record de ventas.

In [ ]:
ventas_productos = facturas.groupby('id_producto')['cantidad_productos'].sum()

mejor_producto = ventas_productos.max()

info_mejor_producto = pd.merge(ventas_productos, productos, on='id_producto')

pd.DataFrame(info_mejor_producto[info_mejor_producto['cantidad_productos'] == mejor_producto])

,id_producto,cantidad_productos,nombre_producto,precio,stock,id_proveedor,fecha_vencimiento
21,22,822,Escritorio de madera,180.0,55,11,2026-06-12 00:00:00


---
### **5.** Muestre el nombre y apellido del cliente que tiene la factura más cara.

In [ ]:
facturas['valor_factura'] = (facturas['cantidad_productos'] * facturas['precio_unitario'])

factura_mas_cara = facturas['valor_factura'].max()

top_factura = facturas[facturas['valor_factura'] == factura_mas_cara]

info_cliente = pd.merge(top_factura, clientes, on='id_cliente')

resultado = info_cliente[['nombre_cliente', 'apellido_cliente']].head(1)
pd.DataFrame(resultado)

,nombre_cliente,apellido_cliente
0,Sofía,Fernández
